> ### ⚠️ Stale numbers — read before trusting cells below
>
> This notebook was authored **before** the April-2026 leakage audit. Any metric it reports
> (e.g. `ROC-AUC ≈ 0.955`, `283,392 rows`, gene-level splits) reflects the **pre-audit**
> pipeline that contained three leakage sources:
>
> 1. Non-missense contamination (null `alt_aa`)
> 2. Definitional circularity via `is_common` / `chr` one-hot
> 3. Paralog leakage across train/test
>
> **The honest numbers live in `results/metrics/` — see `results/metrics/README.md`
> and `leakage_fix_journey.csv`.** Current leak-free baseline:
> `Test ROC-AUC 0.938 [0.935, 0.941]  ·  PR-AUC 0.836 [0.827, 0.843]  ·  ECE 0.015`.
>
> Cells below are **preserved for provenance** but should not be cited in the report.


# Phase 5: Feature Analysis & Dataset Preparation

This notebook documents:
1. Removing redundant features (high correlation)
2. Handling missing values
3. Creating two dataset versions: strict and balanced


In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/merged_clinvar_gnomad_dbnsfp.parquet')
print(f'Input: {len(df):,} rows x {len(df.columns)} columns')


Input: 283,392 rows x 44 columns


## Step 1: Remove Useless Columns


In [2]:
# Drop 100% NaN and zero-variance columns
dropped = []
for col in df.columns:
    if df[col].isna().all():
        dropped.append((col, '100% missing'))
    elif df[col].nunique() <= 1:
        dropped.append((col, 'zero variance'))

print('Columns removed:')
for col, reason in dropped:
    print(f'  {col:<30} -> {reason}')

df = df.drop(columns=[c for c, _ in dropped])
print(f'\nRemaining: {len(df.columns)} columns')


Columns removed:
  MolecularConsequence           -> 100% missing
  has_dbnsfp_features            -> zero variance

Remaining: 42 columns


## Step 2: Remove Highly Correlated Features

When two features have Pearson correlation |r| > 0.95, they carry nearly identical
information. Keeping both adds noise without improving predictive power.

**Rule:** For each correlated pair, we drop the feature with more missing values.
If missing counts are equal, we drop the one that appears later alphabetically
(deterministic tie-break to ensure reproducibility).

The actual pairs and decisions are computed and displayed in the next cell.

In [3]:
import sys
from pathlib import Path
repo_root = Path('..').resolve()
sys.path.insert(0, str(repo_root))

import numpy as np

# Dynamically compute and display correlated pairs from the loaded dataset
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_from_corr = ['pos', 'label', 'review_stars']
feat_cols = [c for c in numeric_cols if c not in exclude_from_corr]

corr = df[feat_cols].corr().abs()
pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]
        if r > 0.95:
            col_a = corr.columns[i]
            col_b = corr.columns[j]
            miss_a = df[col_a].isna().sum()
            miss_b = df[col_b].isna().sum()
            # Drop the feature with more missing values; alphabetical tie-break
            if miss_a > miss_b or (miss_a == miss_b and col_a > col_b):
                drop_col, keep_col = col_a, col_b
            else:
                drop_col, keep_col = col_b, col_a
            pairs.append({'Keep': keep_col, 'Drop': drop_col, 'Correlation': round(r, 4),
                          'Missing (keep)': miss_a if keep_col == col_a else miss_b,
                          'Missing (drop)': miss_b if drop_col == col_b else miss_a})

import pandas as pd
if pairs:
    print('Correlated pairs (|r| > 0.95) — action taken:')
    print(pd.DataFrame(pairs).to_string(index=False))
else:
    print('No highly correlated pairs found (already cleaned).')

Correlated pairs (|r| > 0.95) — action taken:
      Keep      Drop  Correlation  Missing (keep)  Missing (drop)
charge_ref    pI_ref       0.9642           32426           32426
        AF AF_popmax       0.9607               0               0
        AC        AF       0.9680               0               0


## Step 3: Create Two Dataset Versions

| Version | Review Stars | Missing Values | Expected Size |
|---------|-------------|----------------|---------------|
| **Strict** | >= 2 stars | Zero tolerance for conservation | ~37K rows |
| **Balanced** | >= 1 star | Imputed with median | ~283K rows |


In [4]:
# Load final versions and compare
strict = pd.read_parquet('../data/processed/final_strict.parquet')
balanced = pd.read_parquet('../data/processed/final_balanced.parquet')

print('=' * 60)
print('DATASET VERSIONS COMPARISON')
print('=' * 60)
print('{:>20} {:>15} {:>15}'.format('', 'Strict', 'Balanced'))
print('-' * 60)
print('{:>20} {:>15,} {:>15,}'.format('Rows', len(strict), len(balanced)))
print('{:>20} {:>15} {:>15}'.format('Columns', len(strict.columns), len(balanced.columns)))
print('{:>20} {:>15,} {:>15,}'.format('Pathogenic', (strict['label']==1).sum(), (balanced['label']==1).sum()))
print('{:>20} {:>15,} {:>15,}'.format('Benign', (strict['label']==0).sum(), (balanced['label']==0).sum()))
print('{:>20} {:>15.1%} {:>15.1%}'.format('Path. ratio', (strict['label']==1).mean(), (balanced['label']==1).mean()))
print('-' * 60)
print()
print('We will use the BALANCED version for training (more data = better models)')
print('The STRICT version will be used for ablation study comparison')


DATASET VERSIONS COMPARISON
                              Strict        Balanced
------------------------------------------------------------
                Rows          37,670         283,392
             Columns              40              31
          Pathogenic          11,035         132,854
              Benign          26,635         150,538
         Path. ratio           29.3%           46.9%
------------------------------------------------------------

We will use the BALANCED version for training (more data = better models)
The STRICT version will be used for ablation study comparison
